## EDA
Time to get familiar with our data. Let's examine some sample elements of the dataset (this can be re-run multiple times to examine many sample elements of the dataset). Note that the runtime may vary depending on the split randomly chosen, whether the arxiv or pubmed dataset is chosen, and the line number chosen. 

In [ ]:
import itertools
import json
from pathlib import Path
import random

dataset_folder = "../data/raw/"
dataset = "random"
if dataset not in ["arxiv", "pubmed"]:
    dataset = random.choice(["arxiv", "pubmed"])

split = random.choice(["train", "test", "val"])
dataset_path = Path(dataset_folder) / f"{dataset}-dataset/{split}.txt"
with open(dataset_path, "r") as f:
    lines = sum(block.count('\n') for block in iter(lambda: f.read(1024 * 1024), ""))
    print(f"Number of lines in {split} set: {lines}")
    line_choice = random.randint(0, lines - 1)
    
    # Reset file pointer to the beginning of the file and skip to the chosen line
    f.seek(0)
    line = next(itertools.islice(f, line_choice, None))

sample = json.loads(line)
print(f"Sample from {dataset_path} (line {line_choice}):")
print(json.dumps(sample, indent=2))

Number of lines in val set: 6633
Sample from ../data/pubmed-dataset/val.txt (line 1204):
{
  "article_id": "PMC4185323",
  "article_text": [
    "spinal meningiomas are common , comprising 25 - 46% of primary intra - spinal tumors7 ) .",
    "they affect mostly female patients , and the average age of patients is about 50 years7,15,23,24 ) .",
    "spinal meningiomas are located in the intra - dural and extramedullary spaces , and typically adhere to the dura matter7,15 ) . to date , few cases of non - dural - based meningiomas have been reported2,3,4,6,8,9,11,13,14,16,17,18,19,20,21,22 ) , as most are reports of clear cell meningiomas2,3,4,6,8,11,13,14,16,18,20,21,22 ) .",
    "this case report describes the first case of a non - dural based fibrous meningioma in the cervico - thoracic region in a 49-year - old female patient .",
    "a 49-year - old female patient was admitted complaining of persistent pain and numbness in both legs for 4 months and a recently developed gait disturba

The first thing to note is that the article_text and abstract_text fields are indeed lists of strings, frequently but not always ending in a punctuation mark. Perhaps a rough attempt at splitting based on sentences? If the list entry ends in a punctuation mark a space should be added in front of the next line while if the list entry doesn't end in a punctuation mark no space is required.

We can also see that the dataset is all lowercase and separates punctuation with a space from the text (e.g. "gauge theory , physics" instead of "gauge theory, physics"). In the Arxiv dataset there also don't appear to be any contractions with the apostrophe having been replaced by a space. In the PubMed dataset some contractions are present but have an extra space inserted e.g. the patient 's. The final model therefore cannot be expected to use contractions properly, likely instead inserting a space and will use an extra space before punctuation.

The "abstract_text" field for all articles has each list element start with a "`<S>`" tag and end with a "`</S>`" tag.

In the Arxiv dataset we can see a bunch of tags (e.g. @xmath, @xcite, @noop). These appear to be LaTeX macro artifacts, where the converter run on the articles didn't render LaTeX macros as text, instead inserting these tags. Including the @xcite tags may be an interesting approach to training the model to know when a statement needs a citation. Looking at the vocab file in arxiv shows that @xcite is the 27th most common word in the dataset so these tags are quite common. Removing the @xmath tags in math-heavy papers will also cause the model to be shown some sentences that don't make much sense, e.g. "this formula is valid when @xmath53 ;" becomes "this formula is valid when ;". This may become a source of strange behavior by the final model since the text will appear to reference formulas and citations that aren't actually present. There are also some very rare artifacts like "_ standard _" which appears to be related to italics/bold markers, some surviving LaTeX, some surviving citations e.g. "damour & taylor ( 1992 )", and some partially surviving inline equations e.g. "rccc + & & & + & massive ,".

In the PubMed dataset some citations are instead left as normal numbers e.g. "with a reported \n prevalence of 5480%3 , 4 .". This example also shows that some newline characters were left in the PubMed dataset, which can be filtered out since they don't appear in meaningful places. This citation format is much harder to remove since there's no easy way to determine which numbers correspond to a citation and which correspond to an actual number in the text. There's another citation format in this dataset as well, being a number or list of numbers inside brackets e.g. "including hplc [ 28 ] , gc [ 9 , 10 ]" or "gland [ figure 2 ]". This format can be filtered out (and is present in the Arxiv dataset as well). The PubMed dataset also seems to have an incomplete replacement for some figure numbers, with "( fig ." being present as well as "( figure 2 )".

After some consideration we decide to exclude most of the tags so the model can focus on learning from the text itself but include some placeholders for math, citations, and references (see preprocessing.py for details).

Now let's write some code to see exactly how common these different artifacts are

In [ ]:
from IPython.display import display, HTML
import json
from pathlib import Path
import re
from tqdm.notebook import tqdm

dataset_folder = Path("../data/raw/")

# Each pattern is checked against every sentence in article_text of the train split.
# Counts show how many sentences / articles contain at least one match.
# Note: takes ~2.5 minutes to scan the arxiv train set and ~30 seconds to scan the pubmed train set.
artifact_patterns = {
    "@xcite":                   re.compile(r'@xcite'),
    "@xmath*":                  re.compile(r'@xmath\d+'),
    "@noop":                    re.compile(r'@noop'),
    "[ word ] cross-refs":      re.compile(r'\[ [a-zA-Z][^\]]{0,50}\]'),
    "[ N ] numeric citations":  re.compile(r'\[ \d[\d ,]*\]'),
    "_ _ italic markers":       re.compile(r'_ \S.*\S _'),
    "* * bold markers":         re.compile(r'\* \S.*\S \*'),
    r"\n in text":              re.compile(r'\n'),
    "( fig . incomplete refs":  re.compile(r'\( fig \.'),
}

results = {}
for dataset in ["arxiv", "pubmed"]:
    path = dataset_folder / f"{dataset}-dataset/train.txt"
    sent_counts = {name: 0 for name in artifact_patterns}
    art_counts  = {name: 0 for name in artifact_patterns}
    total_sents = 0
    total_arts  = 0

    with open(path) as f:
        for line in tqdm(f, desc=f"Scanning {dataset}", unit=" articles"):
            article = json.loads(line)
            total_arts += 1
            art_hit = {name: False for name in artifact_patterns}
            for sent in article.get("article_text", []):
                total_sents += 1
                for name, pat in artifact_patterns.items():
                    if pat.search(sent):
                        sent_counts[name] += 1
                        art_hit[name] = True
            for name, hit in art_hit.items():
                if hit:
                    art_counts[name] += 1

    results[dataset] = {
        "sent_counts": sent_counts,
        "art_counts":  art_counts,
        "total_sents": total_sents,
        "total_arts":  total_arts,
    }

def pct(n, total):
    return f"{n / total:.1%}"

headers = ["Artifact", "Arxiv sents", "Arxiv sents %", "Arxiv arts", "Arxiv arts %",
           "PubMed sents", "PubMed sents %", "PubMed arts", "PubMed arts %"]

rows = []
for name in artifact_patterns:
    a_s, a_st = results["arxiv"]["sent_counts"][name],  results["arxiv"]["total_sents"]
    a_a, a_at = results["arxiv"]["art_counts"][name],   results["arxiv"]["total_arts"]
    p_s, p_st = results["pubmed"]["sent_counts"][name], results["pubmed"]["total_sents"]
    p_a, p_at = results["pubmed"]["art_counts"][name],  results["pubmed"]["total_arts"]
    rows.append([name, f"{a_s:,}", pct(a_s, a_st), f"{a_a:,}", pct(a_a, a_at),
                       f"{p_s:,}", pct(p_s, p_st), f"{p_a:,}", pct(p_a, p_at)])

th = "style='border:1px solid #ccc; padding:6px 12px; background:#f5f5f5; text-align:center'"
td_l = "style='border:1px solid #ccc; padding:6px 12px; text-align:left'"
td_r = "style='border:1px solid #ccc; padding:6px 12px; text-align:right'"
td_pct = "style='border:1px solid #ccc; padding:6px 12px; text-align:right; color:#666'"

group_style = "style='border:1px solid #ccc; padding:6px 12px; background:#f5f5f5; text-align:center; font-weight:bold'"
html = "<table style='border-collapse:collapse; font-family:monospace; font-size:13px'>"
html += "<tr>"
html += f"<th rowspan=2 {th}>Artifact</th>"
html += f"<th colspan=4 {group_style}>Arxiv</th>"
html += f"<th colspan=4 {group_style}>PubMed</th>"
html += "</tr><tr>"
for label in ["Sentences", "%", "Articles", "%", "Sentences", "%", "Articles", "%"]:
    html += f"<th {th}>{label}</th>"
html += "</tr>"
for row in rows:
    html += "<tr>"
    html += f"<td {td_l}>{row[0]}</td>"
    for i, cell in enumerate(row[1:]):
        style = td_pct if "%" in cell else td_r
        html += f"<td {style}>{cell}</td>"
    html += "</tr>"
html += "</table>"
html += (f"<p style='font-family:monospace; font-size:12px; color:#666; margin-top:8px'>"
         f"Totals — Arxiv: {results['arxiv']['total_sents']:,} sentences, {results['arxiv']['total_arts']:,} articles &nbsp;|&nbsp; "
         f"PubMed: {results['pubmed']['total_sents']:,} sentences, {results['pubmed']['total_arts']:,} articles</p>")
display(HTML(html))

Scanning arxiv: 0 articles [00:00, ? articles/s]

Scanning pubmed: 0 articles [00:00, ? articles/s]